In [1]:
import torch
from torch import nn
from torch.utils.data import DataLoader
from torchvision import datasets
from torchvision.transforms import ToTensor
from torchinfo import summary

In [2]:
train_data = datasets.MNIST(
    download=False,
    root='data',
    transform=ToTensor(),
    train=True,
)
test_data = datasets.MNIST(
    download=False,
    root='data',
    transform=ToTensor(),
    train=False
)

In [3]:
batch_size = 64

train_loader = DataLoader(dataset=train_data, batch_size=batch_size, shuffle=True)
test_loader = DataLoader(dataset=test_data, batch_size=batch_size)

for X, y in train_loader:
    print(f"image (N, C, H, W) {X.shape}")
    print(f"label {y}")
    break

image (N, C, H, W) torch.Size([64, 1, 28, 28])
label tensor([1, 2, 4, 2, 2, 9, 7, 5, 0, 7, 8, 6, 0, 9, 5, 8, 8, 4, 2, 5, 4, 4, 4, 3,
        6, 2, 4, 1, 3, 5, 8, 2, 2, 0, 3, 0, 4, 1, 8, 1, 5, 5, 4, 2, 8, 5, 5, 5,
        0, 8, 9, 5, 3, 3, 5, 4, 6, 9, 9, 9, 1, 0, 5, 9])


In [4]:
device = torch.accelerator.current_accelerator().type if torch.accelerator.is_available() else 'cpu'
print(device)

cuda


In [5]:
class Network(nn.Module):
    def __init__(self):
        super().__init__()
        self.flatten = nn.Flatten()
        self.stack = nn.Sequential(
            nn.Linear(28 * 28, 512),
            nn.ReLU(),
            nn.Linear(512, 512),
            nn.ReLU(),
            nn.Linear(512, 10),
        )
    def forward(self, x):
        x = self.flatten(x)
        pred = self.stack(x)
        return pred
    
model = Network().to(device=device)
summary(model=model)

Layer (type:depth-idx)                   Param #
Network                                  --
├─Flatten: 1-1                           --
├─Sequential: 1-2                        --
│    └─Linear: 2-1                       401,920
│    └─ReLU: 2-2                         --
│    └─Linear: 2-3                       262,656
│    └─ReLU: 2-4                         --
│    └─Linear: 2-5                       5,130
Total params: 669,706
Trainable params: 669,706
Non-trainable params: 0

In [6]:
loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(params=model.parameters(), lr=1e-3)

In [7]:
for X, y in train_loader:
    X, y = X.to(device), y.to(device)
    pred = model(X)
    loss = loss_fn(pred, y)
    print(X.shape)
    print(y)
    print(loss.item())
    break

torch.Size([64, 1, 28, 28])
tensor([2, 9, 3, 8, 9, 3, 3, 9, 4, 9, 2, 6, 0, 3, 4, 9, 0, 1, 8, 4, 1, 1, 2, 7,
        7, 4, 6, 4, 2, 9, 6, 8, 0, 0, 7, 9, 2, 4, 5, 7, 7, 2, 7, 9, 0, 3, 4, 9,
        4, 6, 7, 5, 3, 2, 7, 1, 4, 3, 9, 3, 5, 7, 4, 0], device='cuda:0')
2.308040142059326


In [8]:
def print_fn(item):
    params = [p for p in dir(item)]
    print(params)

In [9]:
def train(datasetloader, model, loss_fn, optimizer):
    size = len(datasetloader.dataset)
    for batch, (X, y) in enumerate(datasetloader):
        X, y = X.to(device), y.to(device)
        model.train()

        # FP
        # calculate error
        pred = model(X)
        loss = loss_fn(pred, y)

        # BP
        loss.backward()
        optimizer.step()
        optimizer.zero_grad()

        if batch % 100 == 0:
            loss, current = loss.item(), (batch + 1) * len(X)
            print(f"loss: {loss:>7f}  [{current:>5d}/{size:>5d}]")


In [10]:
def train(model, loss_fn, optimizer, datasetloader):
    size = len(datasetloader.dataset)
    model.train()
    for batch, (X, y) in enumerate(datasetloader):
        X, y, = X.to(device), y.to(device)
        # FP
        pred = model(X)
        loss = loss_fn(pred, y)
        
        # BP
        loss.backward()
        optimizer.step()
        optimizer.zero_grad()
        
        if batch % 100 == 0:
            loss, current = loss.item(), (batch + 1) * len(X)
            print(f"[loss : {loss:>7f} {current:>5d}/{size:>5d}]")

In [15]:
def test(datasetloader, model, loss_fn):
    size = len(datasetloader.dataset)
    num_batches = len(datasetloader)
    model.eval()
    test_loss, correct = 0,0
    with torch.no_grad():
        for X, y in datasetloader:
            X, y = X.to(device), y.to(device)
            pred = model(X)
            test_loss += loss_fn(pred, y).item()
            correct += (pred.argmax(1) == y).type(torch.float).sum().item()
    test_loss /= num_batches
    correct /= size
    print(f"Test Error : \n Accuracy : {100 * correct:>0.1f}%, Avg Loss : {test_loss:>8f}\n")

In [16]:
epochs = 2
for epoch in range(epochs):
    print(f"epoch {epoch + 1}\n--------")
    train(model, loss_fn, optimizer, train_loader)
    test(test_loader, model, loss_fn)
print("Done!")

epoch 1
--------
[loss : 0.007683    64/60000]
[loss : 0.013928  6464/60000]
[loss : 0.010824 12864/60000]
[loss : 0.127328 19264/60000]
[loss : 0.182348 25664/60000]
[loss : 0.014409 32064/60000]
[loss : 0.008822 38464/60000]
[loss : 0.009385 44864/60000]
[loss : 0.161385 51264/60000]
[loss : 0.018749 57664/60000]
Test Error : 
 Accuracy : 97.7%, Avg Loss : 0.072234

epoch 2
--------
[loss : 0.085105    64/60000]
[loss : 0.004525  6464/60000]
[loss : 0.007441 12864/60000]
[loss : 0.053098 19264/60000]
[loss : 0.030444 25664/60000]
[loss : 0.096413 32064/60000]
[loss : 0.007291 38464/60000]
[loss : 0.006037 44864/60000]
[loss : 0.010931 51264/60000]
[loss : 0.086340 57664/60000]
Test Error : 
 Accuracy : 97.9%, Avg Loss : 0.075455

Done!


In [18]:
torch.save(model.state_dict(), './model.pt')
print('model saved!')

model saved!


In [19]:
model = Network().to(device)
model.load_state_dict(torch.load('./model.pt', weights_only=True))

<All keys matched successfully>

In [35]:
classes = [
    "T-shirt/top",
    "Trouser",
    "Pullover",
    "Dress",
    "Coat",
    "Sandal",
    "Shirt",
    "Sneaker",
    "Bag",
    "Ankle boot",
]

X, y = test_data[0][0], test_data[0][1]
model.eval()
with torch.no_grad():
    X = X.to(device)
    pred = model(X)
    pred, actual = classes[pred[0].argmax(0)], classes[y]
    print(f"predict : {pred}, actual {actual}")


predict : Sneaker, actual Sneaker
